## import

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient
from datetime import datetime
import re
from time import sleep

## connect

In [2]:
client = MongoClient('mongodb://localhost:27017/')
client.drop_database('stock')
db = client['stock']


## collection

In [3]:
collection_ACB = db['ACB']
collection_FPT = db['FPT']
collection_STB = db['STB']
collection_MBB = db['MBB']

### empty 

In [4]:
all_page=[]
stocks=[]

In [5]:
#list of stock to crawl
stock_name = ['ACB','FPT','STB','MBB']

## run webdriver

In [6]:
def crawl_data():
    body = driver.find_element(By.TAG_NAME, "body")
    for _ in range(10):  
        body.send_keys(Keys.END)
        sleep(2)
    # Lấy toàn bộ hàng trong bảng
    rows = driver.find_elements(By.CSS_SELECTOR, ".simplize-table-row.simplize-table-row-level-0")

    for row in rows:
        columns = row.find_elements(By.TAG_NAME, "td")
        _date = columns[0].text
        open_price = columns[1].text.replace(",", "")
        highest_price = columns[2].text.replace(",", "")
        lowest_price = columns[3].text.replace(",", "")
        closing_price = columns[4].text.replace(",", "")
        changed_price = columns[5].text.replace(",", "")
        if (changed_price=='-'):
            changed_price = 0
        price_change_percentage = columns[6].text.replace(",", "")
        if (price_change_percentage == '-'):
            price_change_percentage = 0
        changed_volume = columns[7].text.replace(",", "")
        data = {
            "date": _date,
            "open_price": open_price,
            "highest_price": highest_price,
            "lowest_price": lowest_price,
            "closing_price": closing_price,
            "changed_price": changed_price,
            "price_change_percentage": price_change_percentage,
            "changed_volume": changed_volume
        }
        stocks.append(data)       

In [7]:
for stock in stock_name:
    url = f"https://simplize.vn/co-phieu/{stock}/lich-su-gia"
    driver = webdriver.Chrome()
    driver.get(url)
    sleep(15)  # Đợi trang tải

    try:
        for page_num in range(1, 4):
            print(f"Trang {page_num}")
            crawl_data()  # Gọi hàm crawl
            sleep(10)

            # Tìm và nhấp nút chuyển trang
            next_button = WebDriverWait(driver, 20).until(
                EC.element_to_be_clickable((By.XPATH, 
                    '//*[@id="phan-tich"]/div[2]/div/div/div[2]/div[1]/div/div[3]/ul/li[9]/div'))
            )

            # Cuộn và nhấp vào phần tử
            driver.execute_script("arguments[0].scrollIntoView();", next_button)
            sleep(2)  # Đợi cuộn hoàn tất
            driver.execute_script("arguments[0].click();", next_button)
        if stock == "ACB":
            collection_ACB.insert_many(stocks)
            print(len(stocks))
            stocks = []
        elif stock == "FPT":
            collection_FPT.insert_many(stocks)
            print(len(stocks))
            stocks = []
        elif stock == "STB":
            collection_STB.insert_many(stocks)
            print(len(stocks))
            stocks = []
        elif stock == "MBB":
            collection_MBB.insert_many(stocks)
            print(len(stocks))
            stocks = []
    except Exception as e:
        print(f"Lỗi khi chuyển trang: {e}")
    finally:
        driver.quit()  # Đóng trình duyệt


Trang 1
Trang 2
Trang 3
90
Trang 1
Trang 2
Trang 3
90
Trang 1
Trang 2
Trang 3
90
Trang 1
Trang 2
Trang 3
90


In [8]:
# # crawl each stock
# for i in stock_name:
#     url = "https://simplize.vn/co-phieu/"+ i +"/lich-su-gia"
#     driver = webdriver.Chrome()
#     driver.get(url)
#     sleep(10)
#     # crawl_data()
#     #crawl from each page 
#     a = driver.find_element(By.XPATH , '//*[@id="phan-tich"]/div[2]/div/div/div[2]/div[1]/div/div[3]/ul/li[9]/div')

#     # print(len(a))
#     for i in range(1,4):
#         print(i)
#         sleep(4)
#         crawl_data()
#         page = a.click()
#     #     # wait = WebDriverWait(driver, 20)
#     #     # element = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="phan-tich"]/div[2]/div/div/div[2]/div[1]/div/div[3]/ul/li[9]/div')))
#     #     # element.click()
#     driver.quit()

In [9]:
print(len(stocks))

0


In [10]:
# if stocks:
#     try:
#         collection.insert_many(stocks)
#         print(f"Successfully insert {len(stocks)} stocks.")
#     except Exception as e:
#         print(f"Error insert stocks: {e}")

In [11]:
len(stocks)

0

## close drive

In [12]:
# driver.quit()

## query

In [13]:
for demo in collection.find():
    print(demo)

NameError: name 'collection' is not defined

In [ ]:
for demo in collection.find({'date': '17/10/2024'}):
    print(demo)

In [ ]:
for demo in collection.find().sort('highest_price',-1).limit(1):
    print(demo)

## close DB

In [16]:
# client.close()